# Treino do detector de kanji no Kaggle — YOLO11 + dataset Manga109

Guia para o primeiro treino do detector agnóstico de caracteres usando o dataset `miguelmussalam/manga109-character-bouding-box`.

## Estado atual do dataset (o que veio do Roboflow)

- **17 imagens**, todas em `train/`.
- **3.728 bboxes** anotadas (média ~219 por página, min 61, max 456).
- **1 classe**: `Kanji`.
- Formato YOLO26 (mesmo formato Ultralytics: `class cx cy w h` normalizado).
- `data.yaml` do Roboflow aponta para `../valid/images` e `../test/images` — **essas pastas não existem** no export. Isso precisa ser resolvido antes de treinar.

## O que fazer (visão geral)

1. Anexar o dataset ao notebook Kaggle.
2. Copiar as imagens para `/kaggle/working/`, criar split train/val (14/3) — o `/kaggle/input/` é read-only.
3. Escrever um `data.yaml` novo apontando para os caminhos corretos.
4. Instalar `ultralytics` e rodar o treino do YOLO11n.
5. Inspecionar métricas e detecções visualmente.

## Célula 1 — setup e imports

In [ ]:
import os, shutil, random, yaml
from pathlib import Path

DATASET_SLUG = "manga109-character-bouding-box"
DATASET_OWNER = "miguelmussalam"
DATASET_ROOT = Path(f"/kaggle/input/datasets/{DATASET_OWNER}/{DATASET_SLUG}")

WORK = Path("/kaggle/working/manga")
WORK.mkdir(parents=True, exist_ok=True)

# Confirma que o dataset está anexado e a estrutura bate
print("Root existe?", DATASET_ROOT.exists())
print("Conteúdo do root:")
for p in DATASET_ROOT.rglob("*"):
    if p.is_dir(): print(" ", p.relative_to(DATASET_ROOT))

Se o print mostrar `train/images` e `train/labels`, seguir. Se não, o slug do dataset está diferente — ajuste `DATASET_SLUG`.

## Célula 2 — split train/val

In [ ]:
SRC_IMG = DATASET_ROOT / "train" / "images"
SRC_LBL = DATASET_ROOT / "train" / "labels"

imgs = sorted([p for p in SRC_IMG.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])
print(f"Total de imagens: {len(imgs)}")

random.seed(42)
random.shuffle(imgs)

# 14 treino / 3 validação
val_imgs = set(imgs[:3])
train_imgs = set(imgs[3:])

for split, group in [("train", train_imgs), ("val", val_imgs)]:
    (WORK / split / "images").mkdir(parents=True, exist_ok=True)
    (WORK / split / "labels").mkdir(parents=True, exist_ok=True)
    for img_path in group:
        lbl_path = SRC_LBL / (img_path.stem + ".txt")
        shutil.copy(img_path, WORK / split / "images" / img_path.name)
        if lbl_path.exists():
            shutil.copy(lbl_path, WORK / split / "labels" / lbl_path.name)
        else:
            print(f"AVISO: label ausente para {img_path.name}")

print(f"Treino: {len(list((WORK/'train/images').iterdir()))} imgs")
print(f"Val:    {len(list((WORK/'val/images').iterdir()))} imgs")

## Célula 3 — escrever data.yaml novo

In [ ]:
data_yaml = {
    "path": str(WORK),
    "train": "train/images",
    "val": "val/images",
    "nc": 1,
    "names": ["Kanji"],
}

yaml_path = WORK / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(yaml_path.read_text())

## Célula 4 — instalar Ultralytics e treinar

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# YOLO11n: modelo nano, treina rápido, ideal para primeiro treino
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(WORK / "data.yaml"),
    epochs=150,
    patience=30,          # early stopping
    imgsz=1024,           # kanji é pequeno, precisa resolução alta
    batch=8,              # ajustar para caber na VRAM (T4 = 16GB)
    device=0,
    project="/kaggle/working/runs",
    name="kanji_det_v1",
    amp=True,             # mixed precision
    cache="ram",          # cache dataset em RAM, dataset pequeno cabe
    close_mosaic=15,      # desliga mosaic nas últimas 15 épocas
    plots=True,
    verbose=True,
)

**Notas de hiperparâmetros:**
- `imgsz=1024`: kanji em manga é pequeno (às vezes 20-30px em 1500px de página). 640 pode perder muito, 1024 preserva detalhe. Se der OOM na VRAM, cai para 896 ou 768 antes de reduzir batch.
- `batch=8`: com imgsz=1024 na T4 do Kaggle deve caber. Se OOM, tenta `batch=4`.
- `epochs=150 + patience=30`: dataset pequeno, deixa treinar bastante mas para se estagnar.
- `close_mosaic=15`: mosaic augmentation ajuda no geral mas atrapalha densidade alta de objetos pequenos nas épocas finais.

## Célula 5 — inspecionar métricas

In [ ]:
# resultado principal
print(f"mAP@50:      {results.box.map50:.4f}")
print(f"mAP@50-95:   {results.box.map:.4f}")
print(f"Precision:   {results.box.mp:.4f}")
print(f"Recall:      {results.box.mr:.4f}")

O Ultralytics também salva plots em `/kaggle/working/runs/kanji_det_v1/`:
- `results.png`: curvas de loss e métricas por época
- `val_batch0_pred.jpg`: predições em imagens de validação
- `confusion_matrix.png`: matriz de confusão (com 1 classe, útil para ver ratio FP/FN)

## Célula 6 — inspeção visual em imagens de validação e fora do treino

In [ ]:
from ultralytics import YOLO
import glob

best = YOLO("/kaggle/working/runs/kanji_det_v1/weights/best.pt")

# Inferência nas 3 imagens de validação
val_imgs = glob.glob(str(WORK / "val/images/*"))
results = best.predict(
    val_imgs,
    imgsz=1024,
    conf=0.25,
    save=True,
    project="/kaggle/working/predict",
    name="val_check",
)

# Contagem: quantas detecções por imagem
for r in results:
    print(f"{Path(r.path).name}: {len(r.boxes)} detecções")

Depois, baixa uma página de manga qualquer (fora do Manga109) e roda o mesmo `.predict()` nela. Se detectar razoavelmente bem em página desconhecida, o modelo generaliza. Se não detectar quase nada, tá overfittando nos 14 exemplos de treino.

## O que esperar

Com 14 imagens de treino e 3 de validação, faixa realista:

| mAP@50 | Diagnóstico |
|--------|-------------|
| >0.90  | Desconfia — pode ser vazamento treino/val ou val muito parecido com treino |
| 0.75-0.90 | Bom sinal, dataset consistente. Escala anotação para 50-100 imagens |
| 0.60-0.75 | Normal para esse volume/diversidade. Precisa mais dado ou mais variedade |
| <0.60 | Anotação inconsistente ou split ruim. Não escala anotação ainda, resolve isso primeiro |

**Não olha só o número.** Sempre inspeciona visualmente a saída de `val_batch0_pred.jpg` e as predições da Célula 6. Muitas vezes o mAP diz uma coisa e a inspeção mostra que:
- Detecção tá boa mas quebra em screentones (precisa páginas com screentone no treino)
- Detecção junta 2 kanji verticais em uma bbox (problema de anchor / resolução)
- Detecção perde furigana (esperado, foi decisão ignorar furigana na anotação)

## Próximos passos depois desse baseline

1. Salvar `best.pt` e as métricas em um dataset novo do Kaggle (versionamento).
2. Anotar mais 30-50 imagens **de volumes/autores diferentes** do Manga109 para aumentar diversidade.
3. Retreinar e comparar mAP contra o baseline atual.
4. Em paralelo: começar o gerador sintético do **classificador** de 1233 classes N1, que é independente do detector.